# 第 5 章 パーセプトロン

点を直線で 2 クラスに分けます。誤分類した点だけを使ってモデルを動かすパーセプトロントリックを確かめます。

対応する記事: [第 5 章 パーセプトロン（Jupyter Notebook（Python） の言語版）](../../../docs/article/grokking-machine-learning/python/ch05.md)

実装本体: `apps/grokking-ml-python/src/`

## セットアップ

実装本体（`../src/grokking_ml/`）を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

```bash
cd apps/grokking-ml-python
uv sync
uv run jupyter lab notebooks/
```

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from grokking_ml.ch05_perceptron import *

## データセット

原著と同じ「悲しい／楽しい」文の分類データです。特徴量は単語 `aack` と `beep` の出現回数、ラベルは 1 が「楽しい」です。

In [2]:
points = [(1.0, 0.0), (0.0, 2.0), (1.0, 1.0), (1.0, 2.0),
          (1.0, 3.0), (2.0, 2.0), (2.0, 3.0), (3.0, 2.0)]
labels = [0, 0, 0, 0, 1, 1, 1, 1]

for point, label in zip(points, labels):
    print(f"aack={point[0]:.0f} beep={point[1]:.0f} → {'楽しい' if label else '悲しい'}")

aack=1 beep=0 → 悲しい
aack=0 beep=2 → 悲しい
aack=1 beep=1 → 悲しい
aack=1 beep=2 → 悲しい
aack=1 beep=3 → 楽しい
aack=2 beep=2 → 楽しい
aack=2 beep=3 → 楽しい
aack=3 beep=2 → 楽しい


## トリックは誤分類した点だけを動かす

**正しく分類できている点では、モデルがまったく変わりません。** これが第 6 章のロジスティック回帰との決定的な違いです。

In [3]:
model = Perceptron(weights=(1.0, 2.0), bias=-4.0)
print("元のモデル      ", model)
print("正解した点を渡す", perceptron_trick(model, (1.0, 2.0), label=1, learning_rate=0.1))
print("誤分類の点を渡す", perceptron_trick(model, (1.0, 1.0), label=1, learning_rate=0.1))

元のモデル       Perceptron(weights=(1.0, 2.0), bias=-4.0)
正解した点を渡す Perceptron(weights=(1.0, 2.0), bias=-4.0)
誤分類の点を渡す Perceptron(weights=(1.1, 2.1), bias=-3.9)


## 学習

In [4]:
trained, errors = perceptron_algorithm(points, labels, learning_rate=0.01, epochs=1000, seed=0)

print("重み  ", [round(w, 4) for w in trained.weights])
print(f"バイアス {trained.bias:.4f}")
print(f"正解率  {accuracy(trained, points, labels):.2f}")

重み   [0.02, 0.01]
バイアス -0.0400
正解率  1.00


## パーセプトロン誤差の落とし穴

**初期状態（全パラメータ 0）の誤差は 0 です。** すべての点が境界線上にあるため、誤分類していてもスコアの絶対値が 0 だからです。正解率は 0.5 しかないのに、誤差関数は最良と報告します。

学習の進み具合を見るなら、誤差ではなく **正解率** を見るべきです。

In [5]:
initial = Perceptron(weights=(0.0, 0.0), bias=0.0)
print(f"初期の平均誤差 {mean_perceptron_error(initial, points, labels):.4f}")
print(f"初期の正解率   {accuracy(initial, points, labels):.2f}")
print()
print(f"学習中の最大誤差 {max(errors):.4f}")
print(f"最終の平均誤差   {errors[-1]:.4f}")
print(f"最終の正解率     {accuracy(trained, points, labels):.2f}")

初期の平均誤差 0.0000
初期の正解率   0.50

学習中の最大誤差 0.0325
最終の平均誤差   0.0000
最終の正解率     1.00


## 試してみる

分類では **重みの絶対値に意味がありません**。比率と符号だけが境界線を決めます。すべてを 100 倍しても、予測はまったく同じです。

In [6]:
scaled = Perceptron(
    weights=tuple(w * 100 for w in trained.weights),
    bias=trained.bias * 100,
)
print("元のモデル   ", [trained.predict(p) for p in points])
print("100 倍モデル ", [scaled.predict(p) for p in points])
print("正解         ", labels)

元のモデル    [0, 0, 0, 0, 1, 1, 1, 1]
100 倍モデル  [0, 0, 0, 0, 1, 1, 1, 1]
正解          [0, 0, 0, 0, 1, 1, 1, 1]
